### Manual: 13-batch

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


In [ ]:
"""Input batch: shapes, provenance, and why it is synthetic.

The Kaggle nuScenes mirror has no `nuscenes_infos_*.pkl` yet (open item of ticket
04), so no real sample can be pushed through the MIT pipeline. This cell records
that fact from the mounted dataset and prints the config-shaped synthetic batch
that the forward cell uses instead.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

system = sc.load_json("system.json")
dataset = system.get("dataset", {})
print("dataset root: %s" % dataset.get("root"))
print("infos: %s" % json.dumps(dataset.get("infos")))
print("real sample possible: %s" % dataset.get("real_sample_possible"))

rc, text = sc.run_script("smoke_batch.py", log_name="smoke_batch.log", timeout=900)
batch = sc.load_json("batch.json")
print()
print("rc = %d" % rc)
print("data_source: %s" % batch.get("data_source"))
print("why_synthetic: %s" % batch.get("why_synthetic"))
print("shapes:")
for key in sorted(batch.get("shapes", {})):
    print("  %-18s %s" % (key, json.dumps(batch["shapes"][key])))
sc.update(
    batch={
        "data_source": batch.get("data_source"),
        "shapes": batch.get("shapes"),
        "real_infos_available": bool(dataset.get("real_sample_possible")),
    }
)
print("BATCH_CELL_OK" if "BATCH_OK" in text else "BATCH_CELL_FAILED (see logs/smoke_batch.log)")
